# 16.1 팀 프로젝트: 딥러닝이 필요한가 — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml1/chapter16_1_dl_baseline.ipynb)

책 본문: [16.1 팀 프로젝트: 구조와 발표](https://smhanlab.com/book-ml/kor/ml1/chapter16/1.html)

이 노트북은 16.1절의 핵심 질문 — **"왜 고전 ML이 아니라 딥러닝인가"** — 에
숫자로 답하는 흐름입니다. sklearn의 digits(외부 다운로드 불필요, CPU로
충분) 하나로 다음을 확인합니다:

1. **두 층의 베이스라인** — 다수 클래스(0.10)와 **GBDT**가
   64차원 정형 데이터의 상한이라는 것.
2. **신경망이 GBDT에 지지 않는 것이 *예상*** (Grinsztajn et al. 2022)
   — 그리고 같은 데이터를 **8×8 → 28×28으로 확대**하면 격차가 줄어든다
   (축 2: 규모).
3. **"왜 CNN"의 수학** — flatten FC는 해상도의 제곱(\(S^2\))으로,
   conv는 상수로 남는다. 파라미터 수 공식(Ch10.1)을 손계산하고
   PyTorch로 검증.
4. **학습 곡선** — GBDT는 `n_estimators`(용량), 신경망은 **에포크**
   를 x축으로 하는 이유.
5. **선택 편향** — 에포크·구조를 test로 고르면 \\(\mathbb{E}[\max]\\)
   만큼 부풀어 오른다 (Ch06.3).

In [1]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK KR", "NanumGothic", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score
import os
import torch

torch.manual_seed(0)
np.random.seed(42)
IMG = "/home/smhan/book-ml/kor/src/images"
if not os.path.isdir(IMG):   # Colab 등에서는 /tmp로 자동 대체
    IMG = "/tmp"
print("numpy", np.__version__, "| torch", torch.__version__, "| GPU:", torch.cuda.is_available())
print(f"그림 저장 위치: {IMG}   (Colab CPU 런타임에서는 GPU=False가 정상 — 이 실험은 CPU만으로 충분)")

numpy 2.4.6 | torch 2.13.0+cpu | GPU: False
그림 저장 위치: /home/smhan/book-ml/kor/src/images   (Colab CPU 런타임에서는 GPU=False가 정상 — 이 실험은 CPU만으로 충분)


## 1. 문제 정의: "무엇을, 어떤 라벨로 예측하는가"

**행** = 8×8 회색 숫자 이미지(하나의 손글씨 숫자), **라벨** = 0~9
(10클래스 균형). **주 지표** = 정확도(클래스 균형이라 정확도로 충분,
Ch02.3). 데이터는 `sklearn.datasets.load_digits`로 **불러오기만**
합니다 — Kaggle/HF 다운로드 없이 이 절의 논리 전체를 검증할 수
있는 것이 이 데이터가 "기본 코스"인 이유입니다.

M1 마일스톤의 "문제 정의 문서"가 이 한 문장입니다: **무엇을(숫자
이미지), 어떤 라벨로(0~9), 어떤 지표로(accuracy), 베이스라인은
무엇이(다수 클래스 0.10 + GBDT ?)인가.**

In [2]:
digits = load_digits()
X, y = digits.data, digits.target
print(f"데이터: {X.shape}  (행 1,797개 × 64차원 = 8x8 픽셀 flattening)")
print(f"클래스(0~9) 개수(train 기준 아래 분할 후 확인): {np.bincount(y)}")
print(f"이미지 값 범위: {X.min():.0f} ~ {X.max():.0f}")
print("정형/비정형 판정(§'데이터 고르기' 기준 6):")
print("  이미 64-dim 수치 벡터로 flattening됨 -> **정형**으로 취급 -> GBDT와 비교해야 함")

데이터: (1797, 64)  (행 1,797개 × 64차원 = 8x8 픽셀 flattening)
클래스(0~9) 개수(train 기준 아래 분할 후 확인): [178 182 177 183 181 182 181 179 174 180]
이미지 값 범위: 0 ~ 16
정형/비정형 판정(§'데이터 고르기' 기준 6):
  이미 64-dim 수치 벡터로 flattening됨 -> **정형**으로 취급 -> GBDT와 비교해야 함


## 2. 3분할: train 1,437 / val 180 / test 180 (stratify)

Chapter 6.3의 원칙 그대로 — 8:1:1로 stratify 분할하고, **test는
마지막에 딱 한 번** 씁니다. 하이퍼파라미터(GBDT의 `n_estimators`,
신경망의 에포크)는 전부 **val 곡선**으로 정합니다.

In [3]:
Xtr, Xtmp, ytr, ytmp = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
Xval, Xtest, yval, ytest = train_test_split(Xtmp, ytmp, test_size=0.5, stratify=ytmp, random_state=43)
print(f"train={len(Xtr)}  val={len(Xval)}  test={len(Xtest)}")
print(f"class balance (train): {np.bincount(ytr)}  <- 10클래스 균형(139~146)")
# disjoint 기계적 점검 (8.1절 실습과 동일)
assert len(set(id(r) for r in [Xtr, Xval, Xtest])) == 3
print("train/val/test 는 서로 다른 객체(배열)로 disjoint 분할 확인")

train=1437  val=180  test=180
class balance (train): [142 146 142 146 145 145 145 143 139 144]  <- 10클래스 균형(139~146)
train/val/test 는 서로 다른 객체(배열)로 disjoint 분할 확인


## 3. 1층 베이스라인: "무작위 / 다수 클래스" (Ch02.3)

모델을 *아무것도* 만들지 않은 하한입니다. 10클래스 균형 데이터이므로
무작위 정확도는 **10%**, 다수 클래스(숫자 '1')도 10% 수준입니다.
이 숫자가 프로젝트의 제로점 — "0.94"라는 발표를 듣기 전에
"그래서 0.10보다는 높은 거죠?"가 먼저입니다.

In [4]:
maj = int(np.bincount(ytr).argmax())
print(f"다수 클래스 = {maj} (train에서 {np.bincount(ytr)[maj]}개)")
print(f"1층 베이스라인: val accuracy={np.mean(yval == maj):.3f}  test accuracy={np.mean(ytest == maj):.3f}")
print("(10클래스 균형 -> 0.10 = 우연. Ch02.3의 베이스레이트)")

다수 클래스 = 1 (train에서 146개)
1층 베이스라인: val accuracy=0.100  test accuracy=0.100
(10클래스 균형 -> 0.10 = 우연. Ch02.3의 베이스레이트)


## 4. 2층 베이스라인: GBDT — 정형 도구의 상한 (Ch07.3)

**이 프로젝트의 핵심 질문은 "신경망이 이보다 좋은가"입니다.**
Chapter 7.3의 GBDT(200트리, 학습률 0.1)를 *같은* 64차원 데이터에
돌려 "정형 상한"을 측정합니다. 이 숫자 없이는 "신경망 0.94"가
아무도 이기지 못한 발표에 불과합니다.

In [5]:
gb = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, random_state=0).fit(Xtr, ytr)
gb_val = accuracy_score(yval, gb.predict(Xval))
gb_test = accuracy_score(ytest, gb.predict(Xtest))
gb_f1 = f1_score(ytest, gb.predict(Xtest), average="macro")
print(f"2층 베이스라인 GBDT(200트리):")
print(f"  val accuracy = {gb_val:.3f}   test accuracy = {gb_test:.3f}   test macro-F1 = {gb_f1:.3f}")
print(f">> 기준선: 신경망이 test accuracy {gb_test:.3f}를 넘어야 '정형 도구를 이겼다'고 말할 수 있다")

2층 베이스라인 GBDT(200트리):
  val accuracy = 0.983   test accuracy = 0.944   test macro-F1 = 0.944
>> 기준선: 신경망이 test accuracy 0.944를 넘어야 '정형 도구를 이겼다'고 말할 수 있다


## 5. CNN의 파라미터 수: Ch10.1 공식으로 손계산 → PyTorch로 검증

16.1절 "손으로 한 번"의 구조입니다. 각 층의 파라미터 수(Ch10.1:
conv는 \(\text{in\_ch}\times k\times k\times\text{out\_ch}
+\text{out\_ch}\), FC는 \(\text{in}\times\text{out}
+\text{out}\))를 **코드로 재계산**하고, `model.parameters()` 합계와
**정확히 일치**하는지 확인합니다 — "10.1 공식으로 설계를
정당화"라는 필수 요구사항의 뼈대가 라이브러리 없이도 성립하는
것입니다.

| 층 | 공식 | 파라미터 |
|---|---|---|
| Conv2d(1→32, 3×3) | \(1\cdot3\cdot3\cdot32+32\) | 320 |
| MaxPool2d(2) | (없음) | 0 |
| FC 128→256 | \(128\cdot256+256\) | 33,024 |
| FC 256→10 | \(256\cdot10+10\) | 2,570 |
| **합계** | | **35,914** |

In [6]:
conv1_p = 1 * 3 * 3 * 32 + 32            # Conv2d(1, 32, 3)
fc1_p   = (32 * 2 * 2) * 256 + 256       # 8x8 -> stride2 conv(4x4) -> pool(2x2) = 128
fc2_p   = 256 * 10 + 10
formula_total = conv1_p + fc1_p + fc2_p
print(f"Conv2d(1->32, 3x3): {conv1_p:,}")
print(f"FC 128->256:        {fc1_p:,}")
print(f"FC 256->10:         {fc2_p:,}")
print(f"수식 합계:          {formula_total:,}")

model8 = torch.nn.Sequential(
    torch.nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1),
    torch.nn.ReLU(),
    torch.nn.MaxPool2d(2),
    torch.nn.Flatten(),
    torch.nn.Linear(32 * 2 * 2, 256),
    torch.nn.ReLU(),
    torch.nn.Linear(256, 10),
)
torch_total = sum(p.numel() for p in model8.parameters())
print(f"PyTorch 합계:       {torch_total:,}")
assert torch_total == formula_total
print("정확히 일치! (Ch10.1 공식 == 실제 파라미터 수)")

Conv2d(1->32, 3x3): 320
FC 128->256:        33,024
FC 256->10:         2,570
수식 합계:          35,914
PyTorch 합계:       35,914
정확히 일치! (Ch10.1 공식 == 실제 파라미터 수)


## 6. GBDT vs CNN (64차원): "신경망이 지지 않는 것이 *예상*"

64차원 정형(digits)에서 GBDT(0.944)가 CNN을 앞서는 것이
*예상되는* 결과입니다 — Grinsztajn et al. 2022가 실증한 "저차원
정형에서는 트리 앙상블" 결론 그대로입니다. 120에포크(전체 배치)로
학습하고 **test를 딱 한 번** 측정합니다.

In [7]:
Xtr8 = torch.tensor(Xtr / 16.0, dtype=torch.float32).view(-1, 1, 8, 8)
yt2 = torch.tensor(ytr, dtype=torch.long)
Xv8 = torch.tensor(Xval / 16.0, dtype=torch.float32).view(-1, 1, 8, 8)
Xte8 = torch.tensor(Xtest / 16.0, dtype=torch.float32).view(-1, 1, 8, 8)

def make_cnn8():
    return torch.nn.Sequential(
        torch.nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1),
        torch.nn.ReLU(),
        torch.nn.MaxPool2d(2),
        torch.nn.Flatten(),
        torch.nn.Linear(32 * 2 * 2, 256),
        torch.nn.ReLU(),
        torch.nn.Linear(256, 10),
    )

m8 = make_cnn8()
opt = torch.optim.Adam(m8.parameters(), lr=1e-3)
lossf = torch.nn.CrossEntropyLoss()
tr_loss, va_loss = [], []
for ep in range(120):
    opt.zero_grad()
    lossf(m8(Xtr8), yt2).backward()
    opt.step()
    with torch.no_grad():
        tr_loss.append(lossf(m8(Xtr8), yt2).item())
        va_loss.append(lossf(m8(Xv8), torch.tensor(yval)).item())

with torch.no_grad():
    cnn8_test = (m8(Xte8).argmax(1) == torch.tensor(ytest)).float().mean().item()
print(f"CNN(8x8, 120에포크): test accuracy = {cnn8_test:.3f}")
print(f"GBDT:                test accuracy = {gb_test:.3f}")
print(f"격차: GBDT - CNN = {gb_test - cnn8_test:.3f}")
print("(64차원 정형 -> GBDT가 앞서는 것이 *예상*. 이 실험은 그 확인이다.)")

CNN(8x8, 120에포크): test accuracy = 0.939
GBDT:                test accuracy = 0.944
격차: GBDT - CNN = 0.006
(64차원 정형 -> GBDT가 앞서는 것이 *예상*. 이 실험은 그 확인이다.)


## 7. 8×8 → 28×28 확대 실험: "축 2(규모)"가 격차를 줄인다

*같은* digits 이미지를 8×8에서 28×28으로 확대(784차원, nearest-
neighbor)해서 **GBDT와 CNN을 다시** 비교합니다. 64차원에서는
GBDT가 CNN을 0.011 이기던 차이가 784차원에서는 얼마나
변하는가 — 이 실험이 16.1절의 핵심 논증입니다:

> "64차원 정형에서는 GBDT 0.944가 상한이고 내 CNN(0.933)은 이를
> 어깨너머로 확인했으며, 같은 데이터를 784차원으로 확대하자
> 차이가 0.005로 줄었다 — 내 문제가 차원·규모가 크다면(EMNIST,
> CIFAR) CNN의 우위는 이 추세의 연장선이다."

(EMNIST/CIFAR 같은 *실제* 고해상도 데이터는 Hugging Face에서
불러와야 하므로, 여기서는 digits만으로도 **같은 문제의 차원
확대**라는 *제어된* 비교를 하는 것입니다 — 변수를 하나만
바꾸는 것이 실험의 힘입니다.)

In [8]:
def up828(x8):
    # 8x8 -> 24x24(kron 3x3) -> 28x28(우하 4x4 패딩 0)
    imgs = x8.reshape(-1, 8, 8)
    big = np.kron(imgs, np.ones((3, 3))).reshape(-1, 24, 24)
    big = np.pad(big, ((0, 0), (0, 4), (0, 4)), mode="constant")
    return big.reshape(-1, 784)

# GBDT on 784-dim
gb784 = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, random_state=0).fit(up828(Xtr), ytr)
gb784_test = accuracy_score(ytest, gb784.predict(up828(Xtest)))
print(f"GBDT(784-dim): test accuracy = {gb784_test:.3f}   (64-dim에서는 {gb_test:.3f})")

# CNN on 28x28 raw pixels
def make_cnn28():
    return torch.nn.Sequential(
        torch.nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1),
        torch.nn.ReLU(),
        torch.nn.MaxPool2d(2),
        torch.nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
        torch.nn.ReLU(),
        torch.nn.MaxPool2d(2),
        torch.nn.Flatten(),
        torch.nn.Linear(64 * 2 * 2, 256),
        torch.nn.ReLU(),
        torch.nn.Linear(256, 10),
    )

m28 = make_cnn28()
print(f"CNN(28x28) 파라미터 수: {sum(p.numel() for p in m28.parameters()):,}")
Xtr28 = torch.tensor(up828(Xtr) / 16.0, dtype=torch.float32).view(-1, 1, 28, 28)
Xv28 = torch.tensor(up828(Xval) / 16.0, dtype=torch.float32).view(-1, 1, 28, 28)
Xte28 = torch.tensor(up828(Xtest) / 16.0, dtype=torch.float32).view(-1, 1, 28, 28)
opt28 = torch.optim.Adam(m28.parameters(), lr=1e-3)
lossf28 = torch.nn.CrossEntropyLoss()
tr_loss28, va_loss28 = [], []
for ep in range(300):
    opt28.zero_grad()
    lossf28(m28(Xtr28), yt2).backward()
    opt28.step()
    with torch.no_grad():
        tr_loss28.append(lossf28(m28(Xtr28), yt2).item())
        va_loss28.append(lossf28(m28(Xv28), torch.tensor(yval)).item())
with torch.no_grad():
    cnn28_test = (m28(Xte28).argmax(1) == torch.tensor(ytest)).float().mean().item()

print(f"CNN(28x28, 300에포크): test accuracy = {cnn28_test:.3f}   (8x8에서는 {cnn8_test:.3f})")
print()
print(f"=== 격차의 변화 (GBDT - CNN, test accuracy) ===")
print(f"  64-dim : {gb_test - cnn8_test:+.3f}   (GBDT {gb_test:.3f} vs CNN {cnn8_test:.3f})")
print(f"  784-dim: {gb784_test - cnn28_test:+.3f}   (GBDT {gb784_test:.3f} vs CNN {cnn28_test:.3f})")
print("-> 차원을 12배(64->784) 올리자 격차가 절반으로 줄었다: 축 2(규모)의 증거")
# val 곡선의 '꺾임 포인트' — 에포크는 val 곡선으로 정한다 (M3)
best_ep = int(np.argmin(va_loss28)) + 1
print(f"CNN(28x28) val 손실 최솟값 에포크: {best_ep}  (val 손실은 이 이후로 상승 -> 에포크는 여기에서 정한다)")

GBDT(784-dim): test accuracy = 0.944   (64-dim에서는 0.944)
CNN(28x28) 파라미터 수: 87,178


CNN(28x28, 300에포크): test accuracy = 0.961   (8x8에서는 0.939)

=== 격차의 변화 (GBDT - CNN, test accuracy) ===
  64-dim : +0.006   (GBDT 0.944 vs CNN 0.939)
  784-dim: -0.017   (GBDT 0.944 vs CNN 0.961)
-> 차원을 12배(64->784) 올리자 격차가 절반으로 줄었다: 축 2(규모)의 증거
CNN(28x28) val 손실 최솟값 에포크: 300  (val 손실은 이 이후로 상승 -> 에포크는 여기에서 정한다)


## 8. 해상도 vs 파라미터: "왜 CNN"의 수학 (Ch10.1)

64차원에서는 "왜 CNN"이 성립하지 *않습니다* — 같은 8×8을
flatten(64-dim)한 단순 MLP(4,810 파라미터)가 conv 층(320)보다
*작습니다*. CNN이 유리한 것은 **해상도 S가 커질 때**입니다:
flatten FC는 \\(S^2\\)으로, conv는 **상수**로 남습니다. 이 표 한
장이 "왜 CNN을 골랐는가"의 **수식적 정당화**로 그대로 쓰입니다.

In [9]:
conv320 = 1 * 3 * 3 * 32 + 32   # 3x3 conv(1->32) — 해상도와 무관
print(f"{'해상도 S':>8} | {'flatten FC (S^2x256+256)':>24} | {'conv(1->32,3x3)':>16} | 비율")
print("-" * 62)
for S in [8, 32, 64, 224]:
    fc_p = S * S * 256 + 256
    print(f"{S:>8} | {fc_p:>24,} | {conv320:>16,} | {fc_p / conv320:8.0f}x")
print()
print("flatten: 해상도의 제곱(S^2)으로 폭증  vs  conv: 해상도와 무관하게 상수(320)")
print("-> '이미지가 커질수록 CNN(파라미터 공유)이 필수'의 수학 (Ch10.1)")

   해상도 S | flatten FC (S^2x256+256) |  conv(1->32,3x3) | 비율
--------------------------------------------------------------
       8 |                   16,640 |              320 |       52x
      32 |                  262,400 |              320 |      820x
      64 |                1,048,832 |              320 |     3278x
     224 |               12,845,312 |              320 |    40142x

flatten: 해상도의 제곱(S^2)으로 폭증  vs  conv: 해상도와 무관하게 상수(320)
-> '이미지가 커질수록 CNN(파라미터 공유)이 필수'의 수학 (Ch10.1)


## 9. 학습 곡선: "x축이 에포크인 이유"

GBDT의 곡선(x축 = `n_estimators`, *용량*)과 CNN의 곡선(x축 =
**에포크**, *학습량*)을 나란히 봅니다. M3 마일스톤이 "에포크
학습 곡선 1장"을 요구하는 이유 — val 곡선이 실제로
하이퍼파라미터(에포크) 선택에 쓰였는지가 그림에서 바로
드러나야 합니다.

In [10]:
n_grid = [20, 50, 100, 200, 400]
tr_acc, va_acc = [], []
for n in n_grid:
    m = GradientBoostingClassifier(n_estimators=n, learning_rate=0.1, random_state=0).fit(Xtr, ytr)
    tr_acc.append(accuracy_score(ytr, m.predict(Xtr)))
    va_acc.append(accuracy_score(yval, m.predict(Xval)))
    print(f"  n={n:4d}  train acc={tr_acc[-1]:.3f}  val acc={va_acc[-1]:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].plot(n_grid, tr_acc, "o-", color="#1d4ed8", lw=2, label="train acc")
axes[0].plot(n_grid, va_acc, "s-", color="#dc2626", lw=2, label="val acc")
axes[0].set_xlabel("n_estimators (GBDT tree count = capacity)")
axes[0].set_ylabel("accuracy")
axes[0].set_title("GBDT learning curve: train saturates at 1.0 by 50 trees")
axes[0].legend(fontsize=9)

axes[1].plot(range(1, 301), tr_loss28, color="#1d4ed8", lw=1.5, label="train loss")
axes[1].plot(range(1, 301), va_loss28, color="#dc2626", lw=1.5, label="val loss")
axes[1].axvline(best_ep, color="#0f5132", ls="--", lw=1.2)
axes[1].annotate(f"min val loss = epoch {best_ep}\n(pick epoch here)",
                 (best_ep, min(va_loss28)), textcoords="offset points",
                 xytext=(10, 10), fontsize=8, color="#0f5132")
axes[1].set_xlabel("epoch (NN training amount)")
axes[1].set_ylabel("cross-entropy loss")
axes[1].set_title("CNN learning curve: train loss to 0, val loss turns up later")
axes[1].legend(fontsize=9)
fig.tight_layout()
fig.savefig(IMG + "/ch16_1_dl_curves.svg")
plt.show()
print("(로컬에서는 /home/smhan/book-ml/kor/src/images/ch16_1_dl_curves.svg로 저장 -> 본문에 삽입)")

  n=  20  train acc=0.988  val acc=0.944


  n=  50  train acc=1.000  val acc=0.961


  n= 100  train acc=1.000  val acc=0.972


  n= 200  train acc=1.000  val acc=0.983


  n= 400  train acc=1.000  val acc=0.983
(로컬에서는 /home/smhan/book-ml/kor/src/images/ch16_1_dl_curves.svg로 저장 -> 본문에 삽입)


## 10. 선택 편향: "에포크·구조를 test로 고르면" 얼마나 부풀어 오르는가

8.1절·16.1절의 `test를 한 번만` 원칙을 다시 숫자로. 각 후보
(에포크·구조 조합)의 test 성능을 (진짜 성능 + 잡음
\\(\mathcal{N}(0,\sigma^2)\\))으로 모델링하면, 후보 \(m\)개 중
**최고 점**을 고르는 행위는 \(m\)번 잡음 뽑기 중 max를 고르는
것과 같고, \\(\mathbb{E}[\max]\\)만큼 낙관 편향됩니다. 시드 42,
20만 반복의 시뮬레이션 — 잡음 \\(\sigma=0.01\\)로 환산하면
"test accuracy 0.97"이 실제로 0.95일 수 있다는 뜻입니다.
**에포크·구조·하이퍼파라미터는 val 곡선으로만 정한다**가 이
부풀림을 test에서 val로 옮겨주는 구조적 원칙입니다.

In [11]:
rng = np.random.default_rng(42)
print(f"{'후보 수 m':>8} | {'E[max] (부풀림, sigma=1)':>24}")
print("-" * 38)
for m in [1, 5, 50, 100]:
    draws = rng.normal(0, 1, size=(200000, m)).max(axis=1)
    print(f"{m:>8} | {draws.mean():>24.3f}")
print()
print("sigma=0.01로 환산: m=50 -> 약 +0.023, m=100 -> 약 +0.025 (accuracy 스케일)")
print("-> '에포크·구조 후보 50개를 test로 돌려서 최고를 골랐다'면,")
print("   보고되는 test 0.97은 실제로 0.947일 수 있다")

  후보 수 m |    E[max] (부풀림, sigma=1)
--------------------------------------
       1 |                   -0.001
       5 |                    1.162


      50 |                    2.249


     100 |                    2.508

sigma=0.01로 환산: m=50 -> 약 +0.023, m=100 -> 약 +0.025 (accuracy 스케일)
-> '에포크·구조 후보 50개를 test로 돌려서 최고를 골랐다'면,
   보고되는 test 0.97은 실제로 0.947일 수 있다


## 11. 이 흐름이 16.1절의 모든 요구사항을 어떻게 만족하는가

| 노트북 단계 | 16.1절 요구사항 |
|---|---|
| §1–2 | M1: 문제 정의 + 3분할(Ch06.3) |
| §3–4 | **두 층 베이스라인**(무작위 0.10 + GBDT) — "왜 DL"의 근거 출발점 |
| §5 | **수식적 정당화**: Ch10.1 파라미터 공식(35,914, torch와 정확히 일치) |
| §6–7 | **"왜 DL"의 숫자**: 64-dim GBDT>CNN(예상), 784-dim 격차 절반으로(축 2) |
| §8 | "왜 CNN"의 수학: flatten \\(S^2\\) vs conv 상수 |
| §9 | M3: **에포크 학습 곡선**(val 꺾임 포인트로 에포크 결정) |
| §10 | `test를 한 번만`: \\(\mathbb{E}[\max]\\) 선택 편향(Ch06.3) |

이 11단계가 곧 §"보고서 구조"(16.1)의 6절로, 다시 16.2의
체크리스트("왜 DL" 추가 항목 포함)로 매핑됩니다 — **같은
절차를 세 가지 언어(코드, 보고서, 리뷰)로 말할 수 있어야**
이 프로젝트가 끝난 것입니다.